# 01 · Data Loading & Validation

**Project:** real-time appliance detection from a LoRaWAN / Wi-Fi smart meter (STPM33 on NUCLEO-G071RB).

**Goal of this notebook:** load the recorded dataset, understand its structure, and verify that it is clean and
consistent before any analysis or modelling.

**Appliances monitored:** hair dryer, hair straightener, 200 W bulb, 100 W bulb.

Dataset source: `latestdataset/` of the [LoRaWAN-Smart-Meter](https://github.com/abhigyankesri/LoRaWAN-Smart-Meter)
repository (recorded 21 Nov 2026). The original dataset notes are kept in `data/raw/DATASET_README.md`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.config import APPLIANCES, APPLIANCE_NAMES, DATA_RAW, DATA_PROCESSED, MEASUREMENTS
from src.data_loader import load_labelled, load_events, parse_log

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Raw files

The meter prints one line per sample to its serial console. These captures (`*.log`) were parsed and labelled into
`labelled_dataset.csv`, which is the main file used for modelling. `switch_events.csv` lists every on/off switch.

In [2]:
files = sorted(DATA_RAW.iterdir())
pd.DataFrame({
    "file": [f.name for f in files],
    "size_kB": [round(f.stat().st_size / 1024, 1) for f in files],
})

,file,size_kB
0,alldevicesoff_final_3mins.log,3.3
1,DATASET_README.md,8.2
2,labelled_dataset.csv,122.9
3,lookalikepair.log,14.4
4,pause.log,6.0
5,singlecycles.log,22.7
6,straightnerdata.log,20.7
7,switch_events.csv,6.6
8,teraterm.log,55.3
9,testingdataset.log,19.0


In [3]:
print(open(DATA_RAW / "testingdataset.log", encoding="utf-8").read().splitlines()[0])

[meter] #4000 V=247.21 V  I=0.118 A  P=1.1 W  PF=0.046  f=50.04 Hz  E=700.229 Wh  heap=15368


## 2. Load the labelled dataset

In [4]:
df = load_labelled()
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
df.head()

Rows: 1,370   Columns: 20


,run,source_file,block,split,counter,voltage_V,current_A,power_W,pf,freq_Hz,energy_Wh,hair_dryer,hair_straightener,bulb_200W,bulb_100W,num_devices_on,transition,dryer_medium_setting,combo,time_s
0,1,alldevicesoff_final_3mins.log,A_baseline,train,340,246.93,0.098,-1.0,-0.044,50.06,0.001,0,0,0,0,0,0,0,0000,340
1,1,alldevicesoff_final_3mins.log,A_baseline,train,345,246.82,0.118,-0.6,-0.027,50.06,0.001,0,0,0,0,0,0,0,0000,345
2,1,alldevicesoff_final_3mins.log,A_baseline,train,350,246.89,0.098,1.0,0.043,50.06,0.002,0,0,0,0,0,0,0,0000,350
3,1,alldevicesoff_final_3mins.log,A_baseline,train,355,247.11,0.098,2.2,0.076,50.06,0.002,0,0,0,0,0,0,0,0000,355
4,1,alldevicesoff_final_3mins.log,A_baseline,train,360,247.28,0.098,0.0,0.002,50.06,0.003,0,0,0,0,0,0,0,0000,360


In [5]:
df.dtypes.to_frame("dtype").T

,run,source_file,block,split,counter,voltage_V,current_A,power_W,pf,freq_Hz,energy_Wh,hair_dryer,hair_straightener,bulb_200W,bulb_100W,num_devices_on,transition,dryer_medium_setting,combo,time_s
dtype,int64,category,category,category,int64,float64,float64,float64,float64,float64,float64,int64,int64,int64,int64,int64,int64,int64,object,int64


### Column dictionary

| Group | Column | Meaning |
|---|---|---|
| Metadata | `run` | 1 = morning session, 2 = evening session (board restarted in between) |
| | `source_file` | Raw log the row came from |
| | `block` | Protocol block (`A_baseline`, `B_single_cycles`, `C_straightener`, `D_lookalike`, `pause`, `E_tour1..4`, `F_test`) |
| | `split` | `train` or `test` (test = free household-style use, never used for training) |
| | `counter` | Sample number = seconds since boot (restarts each run). Logs keep every 5th sample |
| Measurements | `voltage_V` | RMS voltage |
| | `current_A` | RMS current (≈20 mA steps, 0.098 A offset at no load) |
| | `power_W` | Active power |
| | `pf` | Power factor |
| | `freq_Hz` | Mains frequency |
| | `energy_Wh` | Energy since boot (cumulative) |
| Targets | `hair_dryer`, `hair_straightener`, `bulb_200W`, `bulb_100W` | 1 = ON, 0 = OFF |
| Helpers | `num_devices_on` | Number of appliances ON |
| | `transition` | 1 for the two samples around each switch (may contain surges) |
| | `dryer_medium_setting` | 1 where the dryer was accidentally on its medium setting (~900 W) |
| Added here | `combo` | 4-digit on/off code: dryer, straightener, 200 W, 100 W (e.g. `0110`) |

## 3. Data quality checks

In [6]:
checks = {
    "missing values": int(df.isna().sum().sum()),
    "duplicate rows": int(df.duplicated().sum()),
    "duplicate (run, counter) keys": int(df.duplicated(["run", "counter"]).sum()),
    "labels not 0/1": int((~df[APPLIANCES].isin([0, 1])).sum().sum()),
    "num_devices_on != sum(labels)": int((df["num_devices_on"] != df[APPLIANCES].sum(axis=1)).sum()),
}
pd.Series(checks, name="count").to_frame()

,count
missing values,0
duplicate rows,0
"duplicate (run, counter) keys",0
labels not 0/1,0
num_devices_on != sum(labels),0


**Sampling continuity.** Inside each log the counter should always step by exactly 5 (no dropped samples), even
though Wi-Fi module error messages appear in some logs.

In [7]:
def step_summary(s):
    vc = s.diff().dropna().astype(int).value_counts()
    return ", ".join(f"step {k}: {v}x" for k, v in vc.items())

counter_range = (df.groupby(["run", "source_file"], observed=True)["counter"]
                   .agg(["min", "max", "count", step_summary])
                   .rename(columns={"step_summary": "counter_steps"}))
counter_range.sort_values(["run", "min"])

min   max  count counter_steps
run source_file                                                   
1   alldevicesoff_final_3mins.log   340   520     37   step 5: 36x
    straightnerdata.log             570  1565    200  step 5: 199x
    lookalikepair.log              1795  2475    137  step 5: 136x
    singlecycles.log               2745  3840    220  step 5: 219x
    pause.log                      3905  4190     58   step 5: 57x
2   teraterm.log                    205  2875    535  step 5: 534x
    testingdataset.log             4000  4910    183  step 5: 182x

In [8]:
df[MEASUREMENTS].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
voltage_V,1370.0,246.111,1.920,240.820,244.700,245.990,247.600,250.160
current_A,1370.0,2.306,2.882,0.079,0.118,0.672,6.054,10.447
power_W,1370.0,556.507,707.870,-3.600,2.850,162.100,1479.300,2392.900
pf,1370.0,0.737,0.425,-0.148,0.115,0.991,1.000,1.000
freq_Hz,1370.0,49.986,0.071,49.760,49.960,50.000,50.020,50.180
energy_Wh,1370.0,275.517,285.612,0.001,39.909,140.514,517.244,920.318


## 4. Verify the log parser

`src/data_loader.parse_meter_line` will also be used on the live Wi-Fi stream, so it must reproduce the labelled
dataset exactly from the raw logs.

In [9]:
parsed = pd.concat([parse_log(p) for p in sorted(DATA_RAW.glob("*.log"))], ignore_index=True)
print(f"Parsed meter samples from logs: {len(parsed):,}")

merged = df.merge(parsed, on=["source_file", "counter"], suffixes=("", "_log"))
cols = ["voltage_V", "current_A", "power_W", "pf", "freq_Hz", "energy_Wh"]
max_diff = {c: float((merged[c] - merged[c + "_log"]).abs().max()) for c in cols}
print(f"Rows matched: {len(merged):,} / {len(df):,}")
pd.Series(max_diff, name="max |labelled - parsed|").to_frame()

Parsed meter samples from logs: 1,370
Rows matched: 1,370 / 1,370


,max |labelled - parsed|
voltage_V,0.0
current_A,0.0
power_W,0.0
pf,0.0
freq_Hz,0.0
energy_Wh,0.0


Every meter sample in the raw logs is present in the labelled CSV, and every value matches exactly (max difference
0.0), so the parser is safe to reuse on the live stream.

## 5. Splits, runs and protocol blocks

In [10]:
block_info = (df.groupby(["run", "block", "split"], observed=True)
                .agg(rows=("counter", "size"), counter_start=("counter", "min"), counter_end=("counter", "max"))
                .reset_index().sort_values(["run", "counter_start"]))
block_info["minutes"] = (block_info["rows"] * 5 / 60).round(1)
block_info

,run,block,split,rows,counter_start,counter_end,minutes
0,1,A_baseline,train,37,340,520,3.1
2,1,C_straightener,train,200,570,1565,16.7
3,1,D_lookalike,train,137,1795,2475,11.4
1,1,B_single_cycles,train,220,2745,3840,18.3
4,1,pause,train,58,3905,4190,4.8
5,2,E_tour1,train,128,205,840,10.7
6,2,E_tour2,train,132,845,1500,11.0
7,2,E_tour3,train,140,1505,2200,11.7
8,2,E_tour4,train,135,2205,2875,11.2
9,2,F_test,test,183,4000,4910,15.2


In [11]:
split_summary = df.groupby("split", observed=True).agg(rows=("counter", "size"))
split_summary["minutes"] = (split_summary["rows"] * 5 / 60).round(1)
split_summary["share"] = (split_summary["rows"] / len(df)).round(3)
split_summary

,rows,minutes,share
split,,,
test,183,15.2,0.134
train,1187,98.9,0.866


## 6. Targets

In [12]:
on_rate = df.groupby("split", observed=True)[APPLIANCES].mean().T.round(3)
on_rate.index = [APPLIANCE_NAMES[a] for a in on_rate.index]
on_rate.columns = [f"ON rate ({c})" for c in on_rate.columns]
on_rate

,ON rate (test),ON rate (train)
Hair dryer,0.372,0.247
Hair straightener,0.607,0.393
200 W bulb,0.765,0.248
100 W bulb,0.579,0.292


In [13]:
combo_counts = pd.crosstab(df["combo"], df["split"])
combo_counts["total"] = combo_counts.sum(axis=1)
print(f"Combinations present: {len(combo_counts)} / 16")
combo_counts

Combinations present: 16 / 16


split,test,train,total
combo,,,
0000,13,342,355
0001,5,112,117
0010,16,69,85
0011,7,33,40
0100,8,232,240
0101,5,42,47
0110,35,32,67
0111,26,32,58
1000,0,69,69


## 7. Switch events

In [14]:
events = load_events()
print(f"Switch events: {len(events)}")
events.head(10)

Switch events: 148


,run,block,counter,appliance,action,source,note
0,1,C_straightener,575,Hair straightener,ON,inferred from data,"cold start; not in notes, first sample showing..."
1,1,C_straightener,1175,Hair straightener,OFF,noted,NaN
2,1,C_straightener,1235,Hair straightener,ON,noted,hot restart 1
3,1,C_straightener,1300,Hair straightener,OFF,noted,NaN
4,1,C_straightener,1360,Hair straightener,ON,noted,hot restart 2
5,1,C_straightener,1425,Hair straightener,OFF,noted,NaN
6,1,C_straightener,1490,Hair straightener,ON,noted,"hot restart 3; noted 1485, step visible at 1490"
7,1,C_straightener,1555,Hair straightener,OFF,noted,NaN
8,1,D_lookalike,1795,100W bulb,ON,inferred from data,noted 1800; bulb already on at the first logge...
9,1,D_lookalike,1845,100W bulb,OFF,noted,NaN


In [15]:
pd.crosstab(events["appliance"], events["action"]).assign(total=lambda t: t.sum(axis=1))

action,OFF,ON,total
appliance,,,
100W bulb,22,22,44
200W bulb,16,16,32
Hair dryer,17,17,34
Hair straightener,19,19,38


In [16]:
events["source"].value_counts().to_frame("events")

,events
source,
noted,145
inferred from data,3


## 8. Save the cleaned dataset

In [17]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
out = DATA_PROCESSED / "dataset_clean.csv"
df.to_csv(out, index=False)
print("Saved", out.relative_to(Path.cwd().parent), df.shape)

Saved data\processed\dataset_clean.csv (1370, 20)


## Summary

- **1,370 labelled samples** (1,187 train / 183 test) at a **5-second** resolution, ≈ 1.9 hours in total.
- No missing values, no duplicate samples, no dropped samples inside any log, and labels are consistent.
- The raw-log parser reproduces the labelled dataset exactly, so it can be reused for the live stream.
- All **16 on/off combinations** of the 4 appliances are present in the training data.
- The test block has a very different mix (appliances are ON much more often), which makes it a realistic but harder
  test.

Next: **02 · Exploratory Data Analysis**.